In [ ]:
# ===== CELL A: HRP helper functions  =====
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

def correl_dist(corr):
    """Distance from correlation matrix (Lopez de Prado)."""
    corr = corr.copy()
    corr = np.clip(corr, -1.0, 1.0)
    return np.sqrt(0.5 * (1 - corr))

def get_quasi_diag(link):
    """Return leaf order after hierarchical clustering."""
    return leaves_list(link)

def get_cluster_var(cov, cluster_items):
    """Compute variance for cluster (cluster_items as list of labels)."""
    sub = cov.loc[cluster_items, cluster_items] if isinstance(cov, pd.DataFrame) else cov[np.ix_(cluster_items, cluster_items)]
    w = np.ones(len(cluster_items)) / len(cluster_items)
    return float(np.dot(w, sub.dot(w)))

def recursive_bisection(cov, sorted_tickers):
    """
    Recursively allocate weights across ordered tickers.
    cov: pd.DataFrame covariance (indexed by tickers)
    sorted_tickers: list of tickers in quasi-diagonal order
    """
    weights = pd.Series(1.0, index=sorted_tickers)
    clusters = [sorted_tickers]
    while clusters:
        new_clusters = []
        for cluster in clusters:
            if len(cluster) == 1:
                continue
            i = len(cluster) // 2
            left = cluster[:i]
            right = cluster[i:]
            var_left = get_cluster_var(cov, left)
            var_right = get_cluster_var(cov, right)
            if (var_left + var_right) > 0:
                alloc_left = 1.0 - var_left / (var_left + var_right)
            else:
                alloc_left = 0.5
            alloc_right = 1.0 - alloc_left
            weights[left] = weights[left] * alloc_left
            weights[right] = weights[right] * alloc_right
            if len(left) > 1:
                new_clusters.append(left)
            if len(right) > 1:
                new_clusters.append(right)
        clusters = new_clusters
    return weights / weights.sum()

def hrp_weights(returns_df, tickers=None, corr_method='pearson', linkage_method='single'):
    """
    Compute HRP weights from returns DataFrame (Date x Ticker).
    Returns: (weights(pd.Series), cov(pd.DataFrame), corr(pd.DataFrame), ordered_tickers(list), linkage_matrix)
    """
    if tickers is not None:
        returns_df = returns_df.reindex(columns=tickers)
    col_means = returns_df.mean(axis=0)
    returns_imputed = returns_df.fillna(col_means)
    cov = returns_imputed.cov()
    corr = returns_imputed.corr(method=corr_method).fillna(0.0)
    dist = correl_dist(corr)
    condensed = squareform(dist.values, checks=False)
    link = linkage(condensed, method=linkage_method)
    order_idx = get_quasi_diag(link)
    ordered_tickers = corr.columns[order_idx].tolist()
    w = recursive_bisection(cov, ordered_tickers)
    return w, cov, corr, ordered_tickers, link


In [ ]:
# ===== CELL B: Compute HRP weights (pure) =====
# Requires pivot_train and tickers from earlier cells

if 'pivot_train' not in globals():
    raise RuntimeError("pivot_train is missing. Run the pivot_train creation cell before this cell.")

# Universe: reuse your `tickers` variable if present, else pivot_train columns
universe = list(tickers) if 'tickers' in globals() else list(pivot_train.columns)
pivot_train2 = pivot_train.reindex(columns=universe)

w_hrp, cov_hrp, corr_hrp, ordered_tickers_hrp, link_hrp = hrp_weights(pivot_train2, tickers=universe, linkage_method='single')
print("✅ HRP weights computed for", len(w_hrp), "tickers.")

# Save HRP weights
out_dir = "/content"
pd.DataFrame({"Ticker": w_hrp.index, "HRP_Weight": w_hrp.values}).to_csv(os.path.join(out_dir, "optimized_weights_hrp.csv"), index=False)
print("Saved HRP weights to:", os.path.join(out_dir, "optimized_weights_hrp.csv"))


In [ ]:
# ===== CELL C: Tilt HRP toward predicted returns (optional) =====
# Uses mu_pred from earlier (mean predicted returns per ticker)

if 'mu_pred' not in globals():
    if 'train_df2' in globals() and 'PredReturn_RF' in train_df2.columns:
        mu_pred = train_df2.groupby('Ticker')['PredReturn_RF'].mean()
    else:
        raise RuntimeError("mu_pred not found. Compute mu_pred before running CELL C.")

mu_pred_aligned = mu_pred.reindex(w_hrp.index).fillna(0.0)
alpha = 0.25   # tune between 0.0 (pure HRP) and 1.0 (pure prediction)

tilt_signal = mu_pred_aligned.clip(lower=0.0)
if tilt_signal.sum() == 0:
    tilt_signal = mu_pred_aligned.abs()
tilt_norm = tilt_signal / (tilt_signal.sum() if tilt_signal.sum() > 0 else 1.0)

w_hrp_tilt = (1 - alpha) * w_hrp + alpha * tilt_norm
w_hrp_tilt = w_hrp_tilt / w_hrp_tilt.sum()

pd.DataFrame({"Ticker": w_hrp_tilt.index, "HRP_Tilt_Weight": w_hrp_tilt.values}).to_csv(os.path.join(out_dir, "optimized_weights_hrp_tilt.csv"), index=False)
print(f"✅ HRP-tilted weights saved (alpha={alpha}) to optimized_weights_hrp_tilt.csv")


In [ ]:
# ===== CELL D: Backtest HRP & HRP-tilt =====
# Requires pivot_test (test period pivot) to exist

if 'pivot_test' not in globals():
    # Recreate pivot_test from test_df (safe aggregate)
    pivot_test = (
        test_df.groupby(["Date", "Ticker"])["DailyReturn"]
        .mean()
        .reset_index()
        .pivot(index="Date", columns="Ticker", values="DailyReturn")
    )
pivot_test = pivot_test.reindex(columns=w_hrp.index).fillna(0.0)

def backtest(weights, pivot_returns):
    R = pivot_returns.fillna(0.0).values
    daily = R.dot(weights)
    if len(daily) == 0:
        return {"Cumulative Return": np.nan, "Annual Return": np.nan, "Annual Volatility": np.nan, "Sharpe": np.nan}
    cum_ret = np.prod(1 + daily) - 1
    ann_ret = (1 + cum_ret) ** (252 / len(daily)) - 1
    vol = np.std(daily) * np.sqrt(252)
    sharpe = (np.mean(daily) * 252) / vol if vol > 0 else 0
    return {"Cumulative Return": cum_ret, "Annual Return": ann_ret, "Annual Volatility": vol, "Sharpe": sharpe}

res_hrp = backtest(w_hrp.values, pivot_test)
res_hrp_tilt = backtest(w_hrp_tilt.values, pivot_test)

print("✅ HRP Backtest Results:")
print("HRP (pure):", res_hrp)
print("HRP (tilted):", res_hrp_tilt)

# Save comparison summary
summary_df = pd.DataFrame([res_hrp, res_hrp_tilt], index=["HRP_Pure", "HRP_Tilted"])
summary_df.to_csv(os.path.join(out_dir, "portfolio_summary_hrp.csv"))
print("Saved portfolio_summary_hrp.csv")
